In [1]:
import os
os.chdir("/home/jovyan/work/MST")  # adjust if needed
os.getcwd()



'/home/jovyan/work/MST'

In [12]:
import torch
from pathlib import Path

from mst.models.dino import DinoV2ClassifierSlice
from mst_xai.xai_methods.gradcam_patch_level import GradCAM_MST
from mst.data.datasets.dataset_3d_odelia import ODELIA_Dataset3D
from mst.inference.predictor import load_model
import torch.nn.functional as F

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

run_dir = Path("/home/jovyan/work/MST/runs")

# run_folder = Path("ODELIA/DinoV2ClassifierSlice_Final")
# path_run = run_dir / run_folder

##For New Model
run_folder = Path("NewModel")
checkpoint_name = "challenge_mstv3-vit_sch_CB_sub2_best.chkpt"
path_run = run_dir / run_folder / checkpoint_name

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("DinoV2ClassifierSlice", path_run, device)
model.to(device)
model.eval()


/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov3_main


DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-11): 12 x SelfAttentionBlock(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): LinearKMaskedBias(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (norm2): Layer

In [6]:
ds = ODELIA_Dataset3D(split="test")

sample = ds[0]   # single sample
uid = sample["uid"]
gt = int(sample["target"])

print("UID:", uid, "GT:", gt)


UID: ODELIA_BRAID1_0246_1_left GT: 0


In [7]:
batch = {
    "source": sample["source"].unsqueeze(0).to(device),
    "target": torch.tensor([gt], device=device)
}


In [8]:
batch["source"].shape
# expected: [1, 3, D, H, W]


torch.Size([1, 1, 32, 224, 224])

In [9]:
logits = model(batch["source"])
pred = logits.argmax(dim=1).item()

print("Predicted class:", pred)


Predicted class: 0


In [10]:
def _register_hooks(self):

    # ---------- Patch-level hook ----------
    # patch_block = self.model.encoder.norm
    patch_block = self.model.encoder.blocks[-1].norm1

    def forward_patch(module, input, output):
        self.patch_activations = output  # (B*D, N, C)

    def backward_patch(module, grad_input, grad_output):
        self.patch_gradients = grad_output[0]

    patch_block.register_forward_hook(forward_patch)
    patch_block.register_full_backward_hook(backward_patch)

    # ---------- Slice-level hook ----------
    slice_block = self.model.slice_fusion.layers[-1].norm1

    def forward_slice(module, input, output):
        self.slice_activations = output  # (B, D, C)

    def backward_slice(module, grad_input, grad_output):
        self.slice_gradients = grad_output[0]

    slice_block.register_forward_hook(forward_slice)
    slice_block.register_full_backward_hook(backward_slice)

In [13]:
# --------------------------------------------------
# Core Grad-CAM
# --------------------------------------------------
def generate(self, batch, target_class: int):

    self.model.zero_grad()
    source = batch["source"].to(self.model.device)

    B, C, D, H, W = source.shape
    #patch_size = 14 if using V2
    patch_size = self.model.encoder.patch_embed.patch_size[0]

    logits = self.model(source, save_attn=False)
    score = logits[:, target_class].sum()
    score.backward()

    # ===============================
    # PATCH-LEVEL CAM
    # ===============================

    acts = self.patch_activations      # (B*D, N, C) # Already test - not zero
    grads = self.patch_gradients


    # ----------------------------------
    # Remove CLS + extra tokens (robust)
    # ----------------------------------

    # detect extra tokens once
    if not hasattr(self, "num_extra_tokens"):
        with torch.no_grad():
            x_enc = source[:1]              # (1,1,D,H,W)
            x_enc = x_enc[:, :, 0]          # take one slice → (1,1,H,W)
            x_enc = x_enc.repeat(1, 3, 1, 1)  # → (1,3,H,W)

            out = self.model.encoder.forward_features(x_enc)

        if "x_storage_tokens" in out:
            self.num_extra_tokens = out["x_storage_tokens"].shape[1] #DinoV3 VitB having 5 extra tokens (CLS + 4 storage tokens)
        elif "x_norm_regtokens" in out:
            self.num_extra_tokens = out["x_norm_regtokens"].shape[1] #DinoV2 VitS having 5 extra tokens (CLS + 4 reg tokens) if it's use register
        else:
            self.num_extra_tokens = 0

    # remove CLS + extra tokens
    acts = acts[:, 1 + self.num_extra_tokens:, :]
    grads = grads[:, 1 + self.num_extra_tokens:, :]

    weights = grads.mean(dim=1)        # (B*D, C)

    cam_patch = (acts * weights.unsqueeze(1)).sum(dim=2)                              # (B*D, N_patches)

    h_p = H // patch_size
    w_p = W // patch_size

    cam_patch = cam_patch.view(B, D, h_p, w_p)


    # ===============================
    # SLICE-LEVEL CAM
    # ===============================

    # remove slice CLS token (there are 33 slices but only 32 have patch-level maps)
    slice_acts = self.slice_activations[:, 1:, :] # (B, 32, 284)
    slice_grads = self.slice_gradients[:, 1:, :]

    slice_weights = slice_grads.mean(dim=1)  # (B, C)

    cam_slice = (
        (slice_acts * slice_weights.unsqueeze(1)).sum(dim=-1)
    )                                        # (B, D)

    cam_slice = cam_slice / (cam_slice.max(dim=1, keepdim=True)[0] + 1e-8)
    assert cam_patch.shape[1] == cam_slice.shape[1]


    # ===============================
    # HIERARCHICAL COMBINATION
    # ===============================

    cam_slice = cam_slice.unsqueeze(-1).unsqueeze(-1)  # (B, D, 1, 1)

    cam_combined = cam_patch * cam_slice               # weight spatial maps
    cam_combined = torch.relu(cam_combined)

    # ===============================
    # Upsample to voxel resolution
    # ===============================

    cam_combined = F.interpolate(
        cam_combined.unsqueeze(1),
        size=(D, H, W),
        mode="trilinear",
        align_corners=False
    ).squeeze(1)

    cam_combined = cam_combined.squeeze(0)

    cam_combined = cam_combined - cam_combined.min()
    cam_combined = cam_combined / (cam_combined.max() + 1e-8)

    return cam_combined

In [14]:
gradcam = GradCAM_MST(model)

In [15]:
sal = gradcam.generate(batch, target_class=pred)

In [16]:
# Inside GradCAM_Slice, but we test externally by re-running backward
model

logits = model(batch["source"])
score = logits[:, pred].sum()
score.backward()

acts = gradcam.activations          # [B, N, C]
grads = gradcam.gradients           # [B, N, C]

print("CLS grad sum:", grads[:, 0, :].abs().sum().item())
print("Slice grad sum:", grads[:, 1:, :].abs().sum().item())


AttributeError: 'GradCAM_MST' object has no attribute 'activations'

# Summary
The classifier depends ONLY on the CLS token.
Slice tokens receive ZERO gradient.

In [ ]:
print("CAM shape:", sal.shape)
print("min / max / std:", sal.min(), sal.max(), sal.std())


CAM shape: torch.Size([32])
min / max / std: tensor(0., device='cuda:0', grad_fn=<MinBackward1>) tensor(0., device='cuda:0', grad_fn=<MaxBackward1>) tensor(0., device='cuda:0', grad_fn=<StdBackward0>)
